In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## HW15 — PCA и t-SNE

Загрузите подготовленные данные Sigma Cabs. Порядок ячеек сохранён; полный CSV не пересчитывался.

In [ ]:
df = pd.read_csv('processed_sigma.csv')
df = df.drop("Trip_ID", axis=1)

X = df.drop("Surge_Pricing_Type", axis=1)
y = df["Surge_Pricing_Type"]

### PCA

### Задание 1

Центрируйте признаки и выделите две главные компоненты PCA. Изобразите проекцию и определите долю объяснённой дисперсии через explained_variance_ratio_.

In [ ]:
### Центрируем данные

### Your code is here

X_centered = X - X.mean()

In [ ]:
from sklearn.decomposition import PCA

### Your code is here
pca = PCA(n_components=2)

pca_dataset = pd.DataFrame(data=pca.fit_transform(X_centered), columns=['PCA1', 'PCA2'])
pca_dataset.head()
# pca.fit_transform(X_centered)


In [ ]:
sns.scatterplot(data=pca_dataset, x="PCA1", y="PCA2")

In [ ]:
pca.explained_variance_ratio_

### Задание 2

Постройте heatmap корреляций исходных признаков с двумя компонентами. Какие связи имеют абсолютную корреляцию не ниже 0.5?

In [ ]:
### Your code is here
first_component_corr = X.corrwith(pca_dataset.PCA1)
second_component_corr = X.corrwith(pca_dataset.PCA2)


corrs = pd.concat((first_component_corr, second_component_corr),
                  axis=1)

corrs.columns = ['PCA1', 'PCA2']

corrs

In [ ]:
import seaborn as sns

fig = plt.figure()

fig.set_size_inches(16, 10)

sns.heatmap(corrs,
            xticklabels=corrs.columns,
            yticklabels=corrs.index,
            cmap='BrBG',
            vmin=-1,
            vmax=1)

plt.show()

### Задание 3

Изобразите 2D-проекцию с метками классов, затем выделите третью компоненту и сделайте 3D-график. Найдите признаки, коррелирующие с третьей компонентой.

In [ ]:
### Your code is here

PCA_dataset = np.concatenate((pca_dataset.values, y.values.reshape(-1, 1)),
                              axis=1)

PCA_dataset = pd.DataFrame(PCA_dataset, columns=['PCA1', 'PCA2', 'CLASS'])


fig = plt.figure()
fig.set_size_inches(16, 10)

sns.scatterplot(data=PCA_dataset, x="PCA1", y="PCA2", hue="CLASS")


Дополнительно сравните попарные корреляции главных компонент.

In [ ]:
### Your code is here
pca3 = PCA(n_components=3)

pca3_dataset = pca3.fit_transform(X_centered)

pca3_dataset = pd.DataFrame(pca3_dataset, columns=['PCA1', 'PCA2', 'PCA3'])
pca3_dataset.head()

first_component_corr_3d = X.corrwith(pca3_dataset.PCA1)
second_component_corr_3d = X.corrwith(pca3_dataset.PCA2)
third_component_corr_3d = X.corrwith(pca3_dataset.PCA3)

corrs_3d = pd.concat((first_component_corr_3d,
                      second_component_corr_3d,
                      third_component_corr_3d), axis=1)

corrs_3d.columns = ['PCA_1', 'PCA_2', 'PCA_3']

corrs_3d

In [ ]:
fig = plt.figure()
fig.set_size_inches(16,10)
sns.heatmap(corrs_3d, xticklabels=corrs_3d.columns,
            yticklabels=corrs_3d.index,
            cmap='BrBG',
            vmin=-1,
            vmax=1)

In [ ]:

PCA_dataset_3d = np.concatenate((pca3_dataset.values, y.values.reshape(-1, 1)),
                                 axis=1)

PCA_dataset_3d = pd.DataFrame(PCA_dataset_3d, columns=['PCA_1', 'PCA_2', 'PCA_3', 'SEGMENT'])

fig = plt.figure()
fig.set_size_inches(16, 10)

ax = plt.axes(projection='3d')

colors = PCA_dataset_3d['SEGMENT'].replace(['A', 'B', 'C', 'D'],
                                            ['orange', 'green', 'red', 'blue'])

ax.scatter3D(PCA_dataset_3d['PCA_1'],
             PCA_dataset_3d['PCA_2'],
             PCA_dataset_3d['PCA_3'],
             c=colors)

### Задание 4

Сравните логистическую регрессию с двумя компонентами и без PCA: подберите гиперпараметры и сравните accuracy/F1 на test. Допустим Pipeline StandardScaler → PCA → LogisticRegression.

In [ ]:
np.random.seed(2022)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_centered, y, test_size=0.2,
                                                    shuffle=True, random_state=2022)

In [ ]:
### Your code is here

pipe = Pipeline([('scaler', StandardScaler()),
                 ('pca', PCA(n_components=2)),
                 ('model', LogisticRegression(solver='saga', l1_ratio=0.5))])

param_grid = {'model__penalty': ['l1', 'l2', 'elasticnet'],
              'model__C': [1e-5, 0.0001, 0.001, 0.01]}

grid = GridSearchCV(estimator=pipe, param_grid=param_grid)

grid.fit(X_train, y_train)



In [ ]:
print(classification_report(y_test, grid.predict(X_test),digits=3))

In [ ]:
pipe_without_pca = Pipeline([('scaler', StandardScaler()),
                 ('model', LogisticRegression(solver='saga', l1_ratio=0.5))])

param_grid = {'model__penalty': ['l1', 'l2', 'elasticnet'],
              'model__C': [1e-5, 0.0001, 0.001, 0.01]}

grid = GridSearchCV(estimator=pipe_without_pca, param_grid=param_grid)

grid.fit(X_train, y_train)

In [ ]:
print(classification_report(y_test, grid.predict(X_test),digits=3))

Проверьте, поможет ли увеличение числа компонент.

### Задание 5

Сравните PCA с 2, 4, 8, 16, 28 компонентами. Используйте L2-регуляризацию и подбор C. Постройте зависимости accuracy и F1 от числа компонент.

In [ ]:
### Чтобы не заставлять вас писать похожий на предыдущий код,
### Написали функцию, которая будет обучать модель для
### произвольного числа n_components главных компонент.

def train_pca_i(n_components, X_train, y_train):
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('pca', PCA(n_components=n_components)),
                     ('model', LogisticRegression(penalty='l2', solver='saga',random_state=1))])

    param_grid = {'model__C': [0.0001, 0.001, 0.01, 0.1]}

    grid = GridSearchCV(pipe, param_grid, cv=5)
    grid.fit(X_train, y_train)

    report = classification_report(y_test, grid.predict(X_test), output_dict=True)
    return (report['accuracy'],
            report['weighted avg']['f1-score'])

In [ ]:
%%time
accuracy_dict, f1_score_dict = {}, {}
components_list = [2, 4, 8, 16, 28]

for n_components in components_list:

    ### Your code is here
    accuracy_dict[n_components], f1_score_dict[n_components] = train_pca_i(n_components, X_train, y_train)


In [ ]:
accuracy_dict

In [ ]:
f1_score_dict

In [ ]:
fig = plt.figure()
fig.set_size_inches(16, 10)

plt.plot(list(accuracy_dict.keys()), list(accuracy_dict.values()))

plt.show()

In [ ]:
fig = plt.figure()
fig.set_size_inches(16, 10)

plt.plot(list(f1_score_dict.keys()), list(f1_score_dict.values()))

plt.show()

Как можно объяснить наблюдаемую зависимость качества от числа компонент? Численные выводы требуют запуска на данных.

### t-SNE

Метод подбирает координаты нового пространства, сопоставляя близость объектов в исходном и новом представлениях.

### Задание 6

Постройте t-SNE с двумя компонентами и раскрасьте объекты по классам. Решение в исходнике не заполнено.

Если обучение слишком затратно, условие допускает ограничить выборку несколькими тысячами объектов. Полный t-SNE здесь не запускался.

In [ ]:
from sklearn.manifold import TSNE

### Your code is here



### Задание 7

Аналогично выполните трёхмерный t-SNE с метками классов. Решение не заполнено.

In [ ]:
### Your code is here



Можно исследовать гиперпараметры t-SNE на небольшой выборке; результаты не предполагаются заранее.